Just a sample of synthetic 2D data representing real-world observations (e.g., people's weight and height), with binary labels indicating whether they will have diabetes.

In [1]:
import numpy as np

# Observations (two features per observation, normalized values for height and weight using z-score)
O = np.array([
    [1.2, 3.4],
    [2.1, 1.5],
    [0.8, 2.7],
    [3.2, 4.1],
    [1.5, 2.2],
    [4.1, 0.9],
    [2.7, 3.6],
    [0.5, 1.8],
    [3.8, 2.4],
    [1.9, 3.1]
])

# Boolean labels
y = np.array([
    True,
    False,
    True,
    False,
    True,
    False,
    True,
    False,
    True,
    False
])

In [6]:

from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)# create 5 stratified folds

knn = KNeighborsClassifier(n_neighbors=7)# 3-NN classifier (3 nearest neighbors)

precisions = []; recalls = []; f_measures = []

for fold, (train_idx, test_idx) in enumerate(skf.split(O, y), start=1):
    O_train, O_test = O[train_idx], O[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    knn.fit(O_train, y_train)# dummy train. simply stores the training observations of all folds except current fold.

    y_pred = knn.predict(O_test) # predict labels of current fold

    # calculate confusion matrix components
    tp = np.sum((y_test == True) & (y_pred == True))
    fp = np.sum((y_test == False) & (y_pred == True))
    fn = np.sum((y_test == True) & (y_pred == False))
    tn = np.sum((y_test == False) & (y_pred == False))

    # preferred order: [[TP, FP], [FN, TN]]
    cm = np.array([
        [tp, fp],
        [fn, tn]
    ])

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f_measure = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0)

    precisions.append(precision)
    recalls.append(recall)
    f_measures.append(f_measure)

    # log the performance this fold
    print(f"\n{'=' * 45}\nFOLD {fold}\n{'=' * 45}")
    print(f"Training observations: {(train_idx)}")
    print(f"Testing observations:  {(test_idx)}")
    print("\nTest predictions:")
    for idx, actual, predicted in zip(test_idx, y_test, y_pred):
        print(f"O[{idx}] = {O[idx]}, Actual = {actual}, Predicted = {predicted}")
    print(f"\nConfusion Matrix [[TP, FP], [FN, TN]]:{cm}")
    print(f"\nTP = {tp}, FP = {fp}")
    print(f"FN = {fn}, TN = {tn}")
    print(f"Fold precision = {precision:.2f}")
    print(f"Fold recall    = {recall:.2f}")

# mean precisions, recalls, f_measures across folds
mean_precision = np.mean(precisions)
mean_recall = np.mean(recalls)
mean_f_measure = np.mean(f_measures)

# Display summary in two rows
print("\n" + "=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)
print("Fold        " + "  ".join(f"{i:>6}" for i in range(1, 6)) + "    Mean")
print("-" * 60)
print("Precision   " + "  ".join(f"{p:6.2f}" for p in precisions) + f"  {mean_precision:6.2f}")
print("Recall      " + "  ".join(f"{r:6.2f}" for r in recalls) + f"  {mean_recall:6.2f}")
print("F-Measure   " + "  ".join(f"{f:6.2f}" for f in f_measures) + f"  {mean_f_measure:6.2f}")



FOLD 1
Training observations: [0 1 3 4 5 6 8 9]
Testing observations:  [2 7]

Test predictions:
O[2] = [0.8 2.7], Actual = True, Predicted = True
O[7] = [0.5 1.8], Actual = False, Predicted = True

Confusion Matrix [[TP, FP], [FN, TN]]:[[1 1]
 [0 0]]

TP = 1, FP = 1
FN = 0, TN = 0
Fold precision = 0.50
Fold recall    = 1.00

FOLD 2
Training observations: [0 1 2 3 4 6 7 9]
Testing observations:  [5 8]

Test predictions:
O[5] = [4.1 0.9], Actual = False, Predicted = False
O[8] = [3.8 2.4], Actual = True, Predicted = True

Confusion Matrix [[TP, FP], [FN, TN]]:[[1 0]
 [0 1]]

TP = 1, FP = 0
FN = 0, TN = 1
Fold precision = 1.00
Fold recall    = 1.00

FOLD 3
Training observations: [0 2 3 4 5 7 8 9]
Testing observations:  [1 6]

Test predictions:
O[1] = [2.1 1.5], Actual = False, Predicted = True
O[6] = [2.7 3.6], Actual = True, Predicted = True

Confusion Matrix [[TP, FP], [FN, TN]]:[[1 1]
 [0 0]]

TP = 1, FP = 1
FN = 0, TN = 0
Fold precision = 0.50
Fold recall    = 1.00

FOLD 4
Training o

Let's see the performance of 10 kNN models, 1NN, 2NN, ..., 10NN, after they do 5-fold internal cross-validations, before the final test.

In [8]:

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

# IMPORTANT: same 5 folds for every k
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
folds = list(skf.split(O, y))

results = []# store mean metrics for each k

for k in range(1, 7):
    precisions, recalls, f_measures = [], [], []

    for train_idx, test_idx in folds:
        O_train, O_test = O[train_idx], O[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(O_train, y_train)
        y_pred = knn.predict(O_test)

        tp = np.sum((y_test == True) & (y_pred == True))
        fp = np.sum((y_test == False) & (y_pred == True))
        fn = np.sum((y_test == True) & (y_pred == False))

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (2 * precision * recall / (precision + recall)if precision + recall else 0.0)

        precisions.append(precision)
        recalls.append(recall)
        f_measures.append(f1)

    results.append([k, np.mean(precisions), np.mean(recalls), np.mean(f_measures)])

# Cross-compare mean metrics
print("\nMean metrics across 5 folds")
print("-" * 53)
print(f"{'k':>3} {'Mean Precision':>16} {'Mean Recall':>13} {'Mean F-measure':>16}")
print("-" * 53)

for k, p, r, f in results: print(f"{k:>3} {p:>16.3f} {r:>13.3f} {f:>16.3f}")

# Identify the k with the highest mean F-measure
best = max(results, key=lambda row: row[3])
print("-" * 53)
print(f"Best k by mean F-measure: {best[0]} (F-measure = {best[3]:.3f})")



Mean metrics across 5 folds
-----------------------------------------------------
  k   Mean Precision   Mean Recall   Mean F-measure
-----------------------------------------------------
  1            0.500         0.800            0.600
  2            0.300         0.400            0.333
  3            0.200         0.400            0.267
  4            0.100         0.200            0.133
  5            0.300         0.600            0.400
  6            0.200         0.200            0.200
-----------------------------------------------------
Best k by mean F-measure: 1 (F-measure = 0.600)
